# Preparar el adaptador fusionado para el Space

Genera el adaptador fusionado de los tres generadores (PEFT, concatenación normalizada `cat --normalizar`, el mismo método que ya se usó en `cuantizacion_colab.ipynb`) y lo deja listo para descargar y subir al Space de Hugging Face, reemplazando el adaptador viejo del Generador 1.

Runtime: **GPU T4**. Duración estimada: ~3-5 min (carga el modelo base una vez, la fusión en sí toma segundos).

In [ ]:
import torch
assert torch.cuda.is_available(), "SIN GPU: cambia el tipo de entorno a T4 GPU"
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
%cd /content
import os
if not os.path.isdir("/content/traductor-jerga-dialectal-slm"):
    !git clone -b develop https://github.com/Anderfg13/traductor-jerga-dialectal-slm.git
%cd /content/traductor-jerga-dialectal-slm
!git pull -q origin develop
!pip install -q transformers peft accelerate
!pip uninstall -y -q torchao

## Paso 1 — Fusionar los tres adaptadores individuales

Mismo método que ya aparece en el paper como "promedio exacto con PEFT, pesos 1/3": 43.5 BLEU / 58.1 chrF, estadísticamente igual a la fusión de mergekit (44.2 / 58.4).

Se borra cualquier intento anterior a medias antes de empezar, para no confundir un archivo viejo con uno nuevo.

In [ ]:
!rm -rf finetuning/checkpoints/peft_cat_norm
!python merging/fusionar_adaptadores.py --metodo cat --normalizar \
    --adaptadores finetuning/checkpoints/generador1/adapter \
                  finetuning/checkpoints/generador2/adapter \
                  finetuning/checkpoints/generador3/adapter \
    --salida finetuning/checkpoints/peft_cat_norm
!ls -la finetuning/checkpoints/peft_cat_norm/fusion/

### Verificación (no seguir si esto falla)

El paso anterior corre como comando de shell (`!...`): si falla, Colab **no detiene** "Ejecutar todas" solo, sigue de largo y el zip del final queda incompleto sin avisar — por eso esta celda verifica a propósito. Si el `assert` falla, sube y lee el error completo que imprimió la celda de arriba (`Paso 1`) antes de seguir; no tiene sentido continuar sin este archivo.

In [ ]:
import os
ruta_pesos = "finetuning/checkpoints/peft_cat_norm/fusion/adapter_model.safetensors"
existe = os.path.exists(ruta_pesos)
print("adapter_model.safetensors existe:", existe)
if existe:
    print("Tamaño:", round(os.path.getsize(ruta_pesos) / 1e6, 1), "MB (debería ser ~15 MB)")
assert existe, "FALTA adapter_model.safetensors — el Paso 1 falló a medias. Lee el error completo en la celda de arriba (Paso 1) antes de continuar."

## Paso 2 — Prueba rápida (opcional pero recomendada)

Traduce unas frases con el adaptador fusionado para confirmar que no está roto antes de subirlo al Space.

In [ ]:
!python evaluation/generar_predicciones.py --adapter finetuning/checkpoints/peft_cat_norm/fusion --n 3

## Paso 3 — Empaquetar y descargar

Descarga **solo** `adapter_config.json` y `adapter_model.safetensors` (lo único que el Space necesita). El `tokenizer.json`, `chat_template.jinja`, etc. del Space NO cambian, son los mismos de siempre.

In [ ]:
import os
ruta_pesos = "finetuning/checkpoints/peft_cat_norm/fusion/adapter_model.safetensors"
assert os.path.exists(ruta_pesos), "FALTA adapter_model.safetensors todavía — no empaques ni descargues un zip a medias. Vuelve al Paso 1."
!cd finetuning/checkpoints/peft_cat_norm/fusion && zip -q -r /content/adapter_fusionado.zip adapter_config.json adapter_model.safetensors
!unzip -l /content/adapter_fusionado.zip
from google.colab import files
files.download("/content/adapter_fusionado.zip")